In [1]:
import json
import cv2
import pycocotools.mask as mask
import os
import subprocess
import numpy as np
import pandas as pd
from skimage.measure import find_contours
import glob

In [2]:
path_to_masks_detections_json = r'path_to\\mask_detections.json' # can be found under yolact/results/
path_to_image_folder = 'path_to_images'

licenses = [{"name": "", "id": 0, "url": ""}]
info = {"contributor": "", "date_created": "", "description": "", "url": "", "version": "", "year": ""}
categories = [{"id": 1, "name": "RBC", "supercategory": ""}, {"id": 2, "name": "WBC", "supercategory": ""}, {"id": 3, "name": "PLT", "supercategory": ""}, {"id": 4, "name": "Background", "supercategory": ""}]
images = []

In [3]:
image_paths = glob.glob(os.path.join(path_to_image_folder, '*.tif')) + glob.glob(os.path.join(path_to_image_folder, '*.jpg')) + glob.glob(os.path.join(path_to_image_folder, '*.png'))
image_id = 1
image_names = [os.path.basename(image) for image in image_paths]
for path in image_paths:
    height, width = cv2.imread(path).shape[:2]
    images.append(
        {
            "id": image_id,
            "width": width,
            "height": height,
            "file_name": os.path.basename(path),
            "license": 0,
            "flickr_url": "",
            "coco_url": "",
            "date_captured": 0
        }
    )
    image_id += 1

In [4]:
annFile=open(path_to_masks_detections_json)
data=json.load(annFile)

converts coco with compressed RLE to coco json with polygon point

In [5]:
new_data = {
    "licenses": licenses,
    "info": info,
    "categories":  categories,
    "images": images, #[{"id": 1, "width": 1000, "height": 1000, "file_name": image_name, "license": 0, "flickr_url": "", "coco_url": "", "date_captured": 0}],
    "annotations": [],
    }

for i in range(len(data)):
    segmentation = []
    rle=data[i]['segmentation']
    maskedArr=mask.decode(rle)
    contours, _ = cv2.findContours(maskedArr, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    valid_poly = 0
  
    for contour in contours:
    # Valid polygons have >= 6 coordinates (3 points)
        if contour.size >= 6:
            segmentation.append(contour.astype(float).flatten().tolist())
            valid_poly += 1
    new_data["annotations"].append(
        {
            "id": int(i+1), 
            "image_id": data[i]["image_id"], 
            "category_id": data[i]["category_id"],
            "segmentation": segmentation,
            "area": float(mask.area(rle)),
            "bbox": list(mask.toBbox(rle)),
            "iscrowd": int(0),
            "attributes": {"occluded": False},
            })

# for saving the coco with decoded RLE
# with open(r'storage_path\\coco_results.json', 'w') as output:
#     json.dump(new_data, output)
# output.close()

converts coco json with polygon points into the format needed for labelme

In [6]:
class CocoDatasetHandler:
    def __init__(self, jsonpath, imgpath):
        ann = jsonpath    

        images = pd.DataFrame.from_dict(ann['images']).set_index('id')
        annotations = pd.DataFrame.from_dict(ann['annotations']).set_index('id')
        categories = pd.DataFrame.from_dict(ann['categories']).set_index('id')

        annotations = annotations.merge(images, left_on='image_id', right_index=True)
        annotations = annotations.merge(categories, left_on='category_id', right_index=True)
        annotations = annotations.assign(
            shapes=annotations.apply(self.coco2shape, axis=1))
        self.annotations = annotations
        self.labelme = {}

        self.imgpath = imgpath
        self.images = pd.DataFrame.from_dict(ann['images']).set_index('file_name')

    def coco2shape(self, row):
        if row.iscrowd == 1:
            shapes = self.rle2shape(row)
        elif row.iscrowd == 0:
            shapes = self.polygon2shape(row)
        return shapes

    def rle2shape(self, row):
        rle, shape = row['segmentation']['counts'], row['segmentation']['size']
        mask = self._rle_decode(rle, shape)
        padded_mask = np.zeros(
            (mask.shape[0]+2, mask.shape[1]+2),
            dtype=np.uint8,
        )
        padded_mask[1:-1, 1:-1] = mask
        points = find_contours(mask, 0.5)
        shapes = [
            [[int(point[1]), int(point[0])] for point in polygon]
            for polygon in points
        ]
        return shapes

    def _rle_decode(self, rle, shape):
        mask = np.zeros([shape[0] * shape[1]], np.bool)
        for idx, r in enumerate(rle):
            if idx < 1:
                s = 0
            else:
                s = sum(rle[:idx])
            e = s + r
            if e == s:
                continue
            assert 0 <= s < mask.shape[0]
            assert 1 <= e <= mask.shape[0], "shape: {}  s {}  e {} r {}".format(shape, s, e, r)
            if idx % 2 == 1:
                mask[s:e] = 1
        # Reshape and transpose
        mask = mask.reshape([shape[1], shape[0]]).T
        return mask

    def polygon2shape(self, row):
        # shapes: (n_polygons, n_points, 2)
        shapes = [
            [[int(points[2*i]), int(points[2*i+1])] for i in range(len(points)//2)]
            for points in row.segmentation
        ]
        return shapes

    def coco2labelme(self):
        fillColor = [255, 0, 0, 128]
        lineColor = [0, 255, 0, 128]

        groups = self.annotations.groupby('file_name')
        for file_idx, (filename, df) in enumerate(groups):
            record = {
                'imageData': None,
                'fillColor': fillColor,
                'lineColor': lineColor,
                'imagePath': filename,
                'imageHeight': int(self.images.loc[filename].height),
                'imageWidth': int(self.images.loc[filename].width),
            }
            record['shapes'] = []

            instance = {
                'line_color': None,
                'fill_color': None,
                'shape_type': "polygon",
            }
            for inst_idx, (_, row) in enumerate(df.iterrows()):
                for polygon in row.shapes:
                    copy_instance = instance.copy()
                    copy_instance.update({
                        'label': row['name'],
                        'group_id': inst_idx,
                        'points': polygon
                    })
                    record['shapes'].append(copy_instance)
            if filename not in self.labelme.keys():
                self.labelme[filename] = record

    def save_labelme(self, file_names, dirpath, save_json_only=True):
        if not os.path.exists(dirpath):
            os.makedirs(dirpath)
        else:
            pass
            #raise ValueError(f"{dirpath} has existed")

        for file in file_names:
            filename = os.path.basename(os.path.splitext(file)[0])
            with open(os.path.join(dirpath, filename+'.json'), 'w') as jsonfile:
                json.dump(self.labelme[file], jsonfile, ensure_ascii=True, indent=2)
            if not save_json_only:
                subprocess.call(['cp', os.path.join(self.imgpath, file), dirpath])


ds = CocoDatasetHandler(new_data, path_to_image_folder)
ds.coco2labelme()
ds.save_labelme(ds.labelme.keys(), path_to_image_folder)